# Pi Agent Benchmark

**17 เคสหลัก + Smoke test** แปลงจาก `deep_agent/2610051200_deepagents_benchmark.ipynb`
ใช้ **Python 3 (ipykernel)** กับ Node bridge ที่เรียก `@earendil-works/pi-coding-agent==0.87.1`
Pi เป็นผู้ควบคุม agent loop; Python ทำหน้าที่ fixtures, business functions, scoring และ export

เปิด Notebook นี้จากโฟลเดอร์ `pi_agent` แล้วรัน **Config → Smoke → แต่ละหมวด** ตามลำดับ
เซลล์ tag `run` เรียกโมเดลจริงและมีค่าใช้จ่าย ส่วน Config ตรวจ import/model catalog เท่านั้น
ไฟล์ที่ส่งมาล้าง outputs แล้ว และยังไม่ได้รัน benchmark จริง

หยุดที่จุด **Restart Kernel** ใน history_003; หลัง restart รันเฉพาะ Config แล้วข้ามไปส่วนกู้ session
ห้าม Run All ข้ามจุดนี้ ประวัติใช้ native JSONL ของ Pi ไม่ใช่หน่วยความจำข้าม process

Web ทั้งสองเคสเป็น `unsupported` สำหรับ integration รุ่นที่ตรวจ จึงไม่เรียกบริการค้นหาอื่นแทน
คะแนน `task_passed` เป็น true / false / null และแยกสถานะผ่าน, ไม่ผ่าน, รอตรวจ, runtime error, unsupported
รายละเอียดและข้อจำกัดอยู่ใน `docs/pi_agents_conversion_notes.md`

# 0.Config

In [2]:
import json
import os
import sys
from copy import deepcopy
from pathlib import Path
from uuid import uuid4

# รองรับทั้งการเปิด Jupyter ที่ repository root และในโฟลเดอร์ pi_agent
PI_DIR = Path.cwd() if (Path.cwd() / 'benchmark_runtime.py').is_file() else Path.cwd() / 'pi_agent'
if not (PI_DIR / 'benchmark_runtime.py').is_file():
    raise FileNotFoundError('เปิด Notebook จาก repository root หรือ pi_agent')
if str(PI_DIR.resolve()) not in sys.path:
    sys.path.insert(0, str(PI_DIR.resolve()))

from benchmark_tools import tool_specs
from benchmark_runtime import (
    SDK, config, shared_root, inspect_runtime, begin_run, run_once, score, show,
    export_results, unsupported_web, apply_review, evidence_hash, digest,
    save_restart_manifest, resume_from_manifest, summarize,
)

CONFIG = config()  # BENCHMARK_MODEL เดิม เช่น openai:gpt-6-luna; ไม่มี fallback
KERNEL_IDENTITY = {'pid': os.getpid(), 'instance': uuid4().hex}
print('Model:', CONFIG['provider'], CONFIG['model'])
print('API key available:', bool(os.getenv(CONFIG['api_key_env'])))
print('Shared root:', shared_root())
print('Thinking level:', CONFIG['thinking_level'])


ValueError: BENCHMARK_MODEL must be provider:model; no implicit provider substitution

In [ ]:
CAPABILITIES = inspect_runtime(CONFIG)  # ไม่เรียกโมเดลและไม่เปิด external MCP
print(json.dumps(CAPABILITIES, ensure_ascii=False, indent=2))
if CAPABILITIES['runtime_status'] != 'completed':
    raise RuntimeError('Pi import/config inspection failed; inspect error above')
if not CAPABILITIES['model_found']:
    print('โมเดลนี้ไม่มีใน Pi catalog; เซลล์ run จะบันทึก runtime_error โดยไม่เปลี่ยนโมเดล')
CONFIG['versions'] = CAPABILITIES['versions']


ทุก run สร้าง ID ก่อนเรียกโมเดล ตั้ง API/harness retry เป็น 0 และปิด compaction/cache warming
budget ใหม่คือ 40 provider requests ต่อ prompt; ไม่เท่ากับ recursion_limit ของ Deep Agents
timeout ต่อ prompt คงตามต้นฉบับ ส่วน Smoke เพิ่ม watchdog 60 วินาที
เวลา `python_wall_s` รวม Node startup และทั้งหมด; `agent_s` รวม model/tool IPC; startup แยกไม่ได้ให้ null

Review ต้องอ้าง `run_id` และ hash ของคำตอบ/หลักฐานรอบปัจจุบัน อย่านำคะแนนจากผลเก่ามากรอก
ดู `run['trace']` สำหรับ tool arguments/results ตามลำดับ; ไม่ใช้ hidden reasoning ในการตรวจ

# 1.Create Pi Agent

### Smoke test

คง `DEEP_AGENTS_SDK_OK` เพราะเป็น fixed token สำหรับ parity ไม่ใช่ชื่อ SDK ที่กำลังรัน
Bridge สร้าง Pi session พร้อม allowlist ว่าง และเก็บ effective system prompt ของ harness ไว้ในผลรัน

In [ ]:
smoke_instructions = (
"This is a connection smoke test. "
        "Do not call tools or delegate work. "
        "Reply with exactly: DEEP_AGENTS_SDK_OK"
)


In [ ]:
smoke_prompt = "Run the smoke test."


In [ ]:
smoke_run = begin_run(CONFIG, 'smoke', smoke_prompt, smoke_instructions,
    category='smoke', tools=[], fixtures={},
    case_version=None, fixture_version=None,
    timeout_s=60, source_recursion_limit=10,
    max_model_requests_per_turn=40)

print("New experiment:", smoke_run['run_id'])


In [ ]:
run_once(smoke_run)
print("Runtime:", smoke_run["runtime_status"])


In [ ]:
score(smoke_run, None, tolerance=0, review_criteria=())
show(smoke_run)


### บันทึกท้ายหมวด

บันทึกได้ทุกสถานะ รวม runtime error และ pending review; export ซ้ำคง run_id เดิม แต่สร้างไฟล์ใหม่

In [ ]:
smoke_export = export_results('smoke', [smoke_run])
print(smoke_export)


# 2.Mathematic calculation

ใช้ calculator ตาม signature เดิม; arithmetic และ ValueError ทำงานใน Python โดยตรง
ตัวเลข tolerance ±0.01, ranking/keys/list order ต้องตรง; bool, NaN, Infinity ไม่ถือเป็นตัวเลขที่ถูกต้อง

In [ ]:
math_tools = tool_specs(['calculator'])
print(json.dumps(math_tools, ensure_ascii=False, indent=2))


In [ ]:
math_instructions = (
"You solve numerical tasks using the provided data. "
        "Choose the appropriate formulas yourself. "
        "Use calculator for arithmetic. "
        "Do not delegate work or use other tools for this task. "
        "Follow the user's output format exactly."
)


### Math 001

In [ ]:
math_prompt = """
บริษัทสมมติ Alpha มีข้อมูลจากงบรวมดังนี้
ตัวเลขทั้งหมดมีหน่วยเป็นล้านบาท:

- รายได้ปี 2024: 1,000
- รายได้ปี 2025: 1,250
- กำไรสุทธิปี 2025: 150
- หนี้สินรวม ณ สิ้นปี 2025: 600
- สินทรัพย์รวม ณ สิ้นปี 2025: 2,000

จงคำนวณ:
1. การเติบโตของรายได้ปี 2025 เทียบกับปี 2024 เป็นเปอร์เซ็นต์
2. อัตรากำไรสุทธิปี 2025 เป็นเปอร์เซ็นต์
3. หนี้สินรวมต่อสินทรัพย์รวม ณ สิ้นปี 2025 เป็นเปอร์เซ็นต์

ข้อกำหนด:
- ใช้ calculator สำหรับการคำนวณ
- ใช้เฉพาะข้อมูลที่ให้
- ตอบเป็น JSON object เท่านั้น ไม่ใส่ Markdown code fence
- มีสาม keys ตามนี้ และค่าต้องเป็นตัวเลข:
  revenue_growth_pct
  net_margin_pct
  liabilities_to_assets_pct
- ค่าเปอร์เซ็นต์ให้ใช้รูปแบบ เช่น 15 สำหรับ 15%
- ปัดผลลัพธ์สุดท้ายเป็นทศนิยมไม่เกินสองตำแหน่ง
"""


In [ ]:
math_001_run = begin_run(CONFIG, 'math_001', math_prompt, math_instructions,
    category='math', tools=math_tools, fixtures={},
    case_version=None, fixture_version=None,
    timeout_s=120, source_recursion_limit=40,
    max_model_requests_per_turn=40)

print("New experiment:", math_001_run['run_id'])


In [ ]:
run_once(math_001_run)
print("Runtime:", math_001_run["runtime_status"])


In [ ]:
expected = {
    "revenue_growth_pct": 25.0,
    "net_margin_pct": 12.0,
    "liabilities_to_assets_pct": 30.0,
}


In [ ]:
score(math_001_run, expected, tolerance=0.01, review_criteria=())
show(math_001_run)


### Math 002

In [ ]:
math_002_prompt = """
บริษัทสมมติสามแห่งมีข้อมูลจากงบรวมดังนี้
ตัวเลขทั้งหมดมีหน่วยเป็นล้านบาท และใช้หลักบัญชีเดียวกัน

Alpha:
- รายได้ปี 2024: 1000
- รายได้ปี 2025: 1250
- กำไรสุทธิปี 2025: 150
- หนี้สินรวม ณ สิ้นปี 2025: 600
- สินทรัพย์รวม ณ สิ้นปี 2025: 2000

Beta:
- รายได้ปี 2024: 800
- รายได้ปี 2025: 880
- กำไรสุทธิปี 2025: 176
- หนี้สินรวม ณ สิ้นปี 2025: 900
- สินทรัพย์รวม ณ สิ้นปี 2025: 1500

Gamma:
- รายได้ปี 2024: 1500
- รายได้ปี 2025: 1800
- กำไรสุทธิปี 2025: 180
- หนี้สินรวม ณ สิ้นปี 2025: 400
- สินทรัพย์รวม ณ สิ้นปี 2025: 2000

งานที่ต้องทำ:
1. คำนวณสามตัวชี้วัดของแต่ละบริษัทเป็นเปอร์เซ็นต์:
   - การเติบโตของรายได้ปี 2025 เทียบกับปี 2024
   - อัตรากำไรสุทธิปี 2025
   - หนี้สินรวมต่อสินทรัพย์รวม ณ สิ้นปี 2025
2. จัดอันดับบริษัท:
   - การเติบโตของรายได้: มากไปน้อย
   - อัตรากำไรสุทธิ: มากไปน้อย
   - หนี้สินรวมต่อสินทรัพย์รวม: น้อยไปมาก
3. คัดบริษัทที่ผ่านเงื่อนไขทั้งสองข้อ:
   - การเติบโตของรายได้ตั้งแต่ 20% ขึ้นไป
   - อัตรากำไรสุทธิตั้งแต่ 12% ขึ้นไป
   เรียงรายชื่อบริษัทที่ผ่านตามตัวอักษร

ข้อกำหนด:
- ใช้ calculator สำหรับการคำนวณ
- ใช้เฉพาะข้อมูลที่ให้
- เปอร์เซ็นต์ใช้ตัวเลข เช่น 15 สำหรับ 15%
- ปัดผลลัพธ์สุดท้ายเป็นทศนิยมไม่เกินสองตำแหน่ง
- ตอบ JSON object เท่านั้น ไม่ใส่ Markdown code fence
- มี top-level keys เท่านั้นดังนี้:

  companies:
    object ที่มี keys Alpha, Beta, Gamma
    แต่ละบริษัทมีสาม numeric keys:
      revenue_growth_pct
      net_margin_pct
      liabilities_to_assets_pct

  rankings:
    object ที่มีสาม keys แต่ละค่าเป็น list ของชื่อบริษัท:
      revenue_growth_desc
      net_margin_desc
      liabilities_to_assets_asc

  eligible_companies:
    list ของชื่อบริษัทที่ผ่านเงื่อนไข
"""


In [ ]:
math_002_run = begin_run(CONFIG, 'math_002', math_002_prompt, math_instructions,
    category='math', tools=math_tools, fixtures={},
    case_version=None, fixture_version=None,
    timeout_s=180, source_recursion_limit=80,
    max_model_requests_per_turn=40)

print("New experiment:", math_002_run['run_id'])


In [ ]:
run_once(math_002_run)
print("Runtime:", math_002_run["runtime_status"])


In [ ]:
math_002_expected = {
    "companies": {
        "Alpha": {
            "revenue_growth_pct": 25.0,
            "net_margin_pct": 12.0,
            "liabilities_to_assets_pct": 30.0,
        },
        "Beta": {
            "revenue_growth_pct": 10.0,
            "net_margin_pct": 20.0,
            "liabilities_to_assets_pct": 60.0,
        },
        "Gamma": {
            "revenue_growth_pct": 20.0,
            "net_margin_pct": 10.0,
            "liabilities_to_assets_pct": 20.0,
        },
    },
    "rankings": {
        "revenue_growth_desc": ["Alpha", "Gamma", "Beta"],
        "net_margin_desc": ["Beta", "Alpha", "Gamma"],
        "liabilities_to_assets_asc": ["Gamma", "Alpha", "Beta"],
    },
    "eligible_companies": ["Alpha"],
}


In [ ]:
score(math_002_run, math_002_expected, tolerance=0.01, review_criteria=())
show(math_002_run)


### Math 003

In [ ]:
math_003_prompt = """
บริษัทสมมติ Delta มีข้อมูลจากงบรวม:

- รายได้ปี 2024: 0 ล้านบาท
- รายได้ปี 2025: 1.5 พันล้านบาท
- กำไรสุทธิปี 2025: ไม่มีข้อมูล
- หนี้สินรวม ณ สิ้นปี 2025: 600 ล้านบาท
- สินทรัพย์รวม ณ สิ้นปี 2025: 2 พันล้านบาท

จง:
1. แปลงรายได้ปี 2025 เป็นล้านบาท
2. คำนวณการเติบโตของรายได้ปี 2025 เทียบปี 2024 เป็นเปอร์เซ็นต์
3. คำนวณอัตรากำไรสุทธิปี 2025 เป็นเปอร์เซ็นต์
4. คำนวณหนี้สินรวมต่อสินทรัพย์รวมเป็นเปอร์เซ็นต์

ข้อกำหนด:
- ใช้ calculator สำหรับการคำนวณและแปลงหน่วย
- ข้อมูลขาด ห้ามสมมติตัวเลขขึ้นมา
- ค่าที่คำนวณไม่ได้ให้เป็น null ไม่ใช้ 0 หรือ Infinity แทน
- ระบุ reason_code ของแต่ละอัตราส่วน:
  OK = คำนวณได้
  ZERO_DENOMINATOR = ตัวหารเป็นศูนย์
  MISSING_DATA = ข้อมูลที่จำเป็นขาด
- เปอร์เซ็นต์ใช้ตัวเลข เช่น 15 สำหรับ 15%
- ตอบ JSON เท่านั้น ไม่ใส่ Markdown code fence
- ใช้โครงสร้างดังนี้ โดยแทนที่ข้อความตัวอย่างด้วยค่าจริง:

{
  "revenue_2025_million": "ตัวเลข",
  "revenue_growth": {
    "value_pct": "ตัวเลขหรือ null",
    "reason_code": "รหัส"
  },
  "net_margin": {
    "value_pct": "ตัวเลขหรือ null",
    "reason_code": "รหัส"
  },
  "liabilities_to_assets": {
    "value_pct": "ตัวเลขหรือ null",
    "reason_code": "รหัส"
  }
}
"""


In [ ]:
math_003_run = begin_run(CONFIG, 'math_003', math_003_prompt, math_instructions,
    category='math', tools=math_tools, fixtures={},
    case_version=None, fixture_version=None,
    timeout_s=120, source_recursion_limit=40,
    max_model_requests_per_turn=40)

print("New experiment:", math_003_run['run_id'])


In [ ]:
run_once(math_003_run)
print("Runtime:", math_003_run["runtime_status"])


In [ ]:
math_003_expected = {
    "revenue_2025_million": 1500.0,
    "revenue_growth": {
        "value_pct": None,
        "reason_code": "ZERO_DENOMINATOR",
    },
    "net_margin": {
        "value_pct": None,
        "reason_code": "MISSING_DATA",
    },
    "liabilities_to_assets": {
        "value_pct": 30.0,
        "reason_code": "OK",
    },
}


In [ ]:
score(math_003_run, math_003_expected, tolerance=0.01, review_criteria=())
show(math_003_run)


### สำรวจสูตรที่เลือกจากรอบปัจจุบัน

พิจารณา operation/a/b และผลเครื่องมือจริง ไม่มีข้อสรุป failure analysis จากรอบ Deep Agents ปะปน

In [ ]:
print(json.dumps([e for e in math_002_run.get('trace', []) if e['kind'] in ('tool_call', 'tool_result')], ensure_ascii=False, indent=2))


### บันทึกท้ายหมวด

บันทึกได้ทุกสถานะ รวม runtime error และ pending review; export ซ้ำคง run_id เดิม แต่สร้างไฟล์ใหม่

In [ ]:
math_export = export_results('math', [math_001_run, math_002_run, math_003_run])
print(math_export)


# 3.MCP

FastMCP fixture เดิมเชื่อมผ่าน **stdio MCP จริง** กับ `pi-mcp-adapter==3.2.0` แบบ host-managed
Adapter เป็น dependency เพิ่ม ไม่ใช่ความสามารถ built-in ของ Pi; discovery เสร็จก่อนเริ่มจับเวลา agent
Adapter ปกติเติม prefix `finance_`; bridge map กลับชื่อเดิมโดยไม่แก้ arguments หรือผลลัพธ์
ปิด client/server ใน finally ทุกครั้ง; transport ต่างจาก in-memory ในต้นฉบับ
Business retry เป็นการตัดสินใจของ Pi ตาม error flag; transport ไม่ resend คำขอเอง

In [ ]:
COMPANY = {
    "company_id": "cmp_a17",
    "company_name": "Alpha",
}

STATEMENT = {
    "company_id": "cmp_a17",
    "year": 2025,
    "revenue_million": 1375,
    "net_profit_million": 165,
    "source_id": "fixture_alpha_2025_v1",
}


In [ ]:
mcp_instructions = (
"Answer using the provided financial data tools. "
                "Resolve the company before retrieving its statement. "
                "Do not guess identifiers or financial figures. "
                "Do not delegate or use other tools. "
                "Follow the requested JSON format."
)


In [ ]:
mcp_fixtures = {'COMPANY': COMPANY, 'STATEMENT': STATEMENT}


### MCP 001

In [ ]:
mcp_prompt = """
หาข้อมูลงบปี 2025 ของบริษัท Alpha จากเครื่องมือที่ให้

ตอบ JSON object เท่านั้น มี keys:
- company_id
- year
- revenue_million
- net_profit_million
- source_id

ใช้ข้อมูลจากผลเครื่องมือจริง ห้ามเดารหัสบริษัทหรือตัวเลข
ไม่ใส่ Markdown code fence
"""


In [ ]:
mcp_001_run = begin_run(CONFIG, 'mcp_001', mcp_prompt, mcp_instructions,
    category='mcp', tools=[], fixtures=mcp_fixtures,
    case_version=None, fixture_version=None,
    timeout_s=120, source_recursion_limit=40,
    max_model_requests_per_turn=40)
mcp_001_run['mcp'] = {'retry_fixture': False}
mcp_001_run['implementation_differences'].append('pi-mcp-adapter host-managed; stdio instead of in-memory; finance_ names mapped back to originals')
print("New experiment:", mcp_001_run['run_id'])


In [ ]:
run_once(mcp_001_run)
print("Runtime:", mcp_001_run["runtime_status"])


In [ ]:
score(mcp_001_run, STATEMENT, tolerance=0, review_criteria=())
show(mcp_001_run)


### MCP 002

In [ ]:
mcp_002_prompt = """
ค้นหาบริษัท Omega จากเครื่องมือที่ให้ และดึงงบปี 2025 หากพบบริษัท

ข้อกำหนด:
- ห้ามเดา company ID หรือใช้ข้อมูลของบริษัทอื่นแทน
- หากไม่พบบริษัท ให้รายงานว่าไม่พบและไม่เรียกดึงงบ
- ตอบ JSON object เท่านั้น ไม่ใส่ Markdown code fence

โครงสร้างคำตอบ:
{
  "status": "found หรือ not_found",
  "company_name": "Omega",
  "statement": "ข้อมูลงบจากเครื่องมือ หรือ null หากไม่พบ"
}
"""


In [ ]:
mcp_002_run = begin_run(CONFIG, 'mcp_002', mcp_002_prompt, mcp_instructions,
    category='mcp', tools=[], fixtures=mcp_fixtures,
    case_version=None, fixture_version=None,
    timeout_s=120, source_recursion_limit=40,
    max_model_requests_per_turn=40)
mcp_002_run['mcp'] = {'retry_fixture': False}
mcp_002_run['implementation_differences'].append('pi-mcp-adapter host-managed; stdio instead of in-memory; finance_ names mapped back to originals')
print("New experiment:", mcp_002_run['run_id'])


In [ ]:
run_once(mcp_002_run)
print("Runtime:", mcp_002_run["runtime_status"])


In [ ]:
expected_002 = {
    "status": "not_found",
    "company_name": "Omega",
    "statement": None,
}


In [ ]:
score(mcp_002_run, expected_002, tolerance=0, review_criteria=())
show(mcp_002_run)


### MCP 003

In [ ]:
mcp_003_prompt = """
หาข้อมูลงบปี 2025 ของบริษัท Alpha จากเครื่องมือที่ให้

นโยบายจัดการข้อผิดพลาด:
- หาก tool คืน error ที่ retryable เป็น true ให้ลองคำขอนั้นใหม่
  ได้ไม่เกินหนึ่งครั้ง
- หากยังไม่สำเร็จ ให้รายงานข้อผิดพลาดตามจริง
- ห้ามเดารหัสบริษัทหรือตัวเลข

เมื่อสำเร็จ ตอบ JSON object เท่านั้น มี keys:
- company_id
- year
- revenue_million
- net_profit_million
- source_id

ไม่ใส่ Markdown code fence
"""


In [ ]:
mcp_003_run = begin_run(CONFIG, 'mcp_003', mcp_003_prompt, mcp_instructions,
    category='mcp', tools=[], fixtures=mcp_fixtures,
    case_version=None, fixture_version=None,
    timeout_s=120, source_recursion_limit=40,
    max_model_requests_per_turn=40)
mcp_003_run['mcp'] = {'retry_fixture': True}
mcp_003_run['implementation_differences'].append('pi-mcp-adapter host-managed; stdio instead of in-memory; finance_ names mapped back to originals')
print("New experiment:", mcp_003_run['run_id'])


In [ ]:
run_once(mcp_003_run)
print("Runtime:", mcp_003_run["runtime_status"])


In [ ]:
score(mcp_003_run, STATEMENT, tolerance=0, review_criteria=())
show(mcp_003_run)


### บันทึกท้ายหมวด

บันทึกได้ทุกสถานะ รวม runtime error และ pending review; export ซ้ำคง run_id เดิม แต่สร้างไฟล์ใหม่

In [ ]:
mcp_export = export_results('mcp', [mcp_001_run, mcp_002_run, mcp_003_run])
print(mcp_export)


# 4.Web search

### ข้อจำกัด native hosted web ในรุ่นที่ติดตั้ง

source/types ของ Pi Responses adapter ไม่ส่ง `web_search_call.action` และ URL annotations เป็น public events
แม้แก้ payload ได้ ก็ยังตรวจ search/open_page/citations ไม่ครบ จึงไม่ inject `web_search` แบบเดา schema
สองเคสนี้เก็บ prompt เดิมและสถานะ unsupported โดยไม่มี model/tool call

ถ้ารุ่นในอนาคตรองรับ ต้องตรวจ search แยกจาก open_page; citation ไม่ใช่หลักฐานเปิดหน้า
เคส 001 ต้องมีแหล่ง Microsoft; เคส 002 ต้องมีทั้ง Microsoft และ Alphabet
สอง URL ของ Microsoft ไม่พอ และ sec.gov เพียงอย่างเดียวบอกบริษัทไม่ได้
การเปิดสำเร็จและเนื้อหารองรับคำตอบต้องมีหลักฐาน มิฉะนั้น pending_review

In [ ]:
native_web_system_prompt = """
You are a financial research assistant.

Search the web for official sources, then open relevant URLs
found during search using the web tool's page-opening capability.
Read the opened sources before analyzing the figures.
Do not rely only on search snippets.

For comparisons between companies, open supporting sources
for each company.

Prefer official company publications or filings submitted to the SEC.
Distinguish fiscal years from calendar years and quarters.
Cite sources supporting your claims.
If you cannot open a source or evidence is insufficient,
state the limitation. Do not invent data or claim a page was read.

Do not delegate or use other tools for this task.
Answer in Thai.
""".strip()


### Web 001

In [ ]:
native_web_prompt = """
ค้นเว็บเพื่อเปรียบเทียบรายได้รวมของ Microsoft
สำหรับปีงบประมาณ 2024 และ 2023

ขั้นตอนที่ต้องทำ:
- ค้นหาแหล่งข้อมูลทางการของ Microsoft
- เปิดอ่าน URL ที่พบจากการค้นหาอย่างน้อยหนึ่งหน้า
- ตรวจตัวเลขจากเนื้อหาที่เปิดอ่าน ไม่ใช้เพียงข้อความย่อผลค้นหา

ข้อกำหนด:
1. ใช้รายได้รวมทั้งปี ไม่ใช้รายไตรมาสหรือราย segment
2. แสดงตัวเลขเป็นล้านดอลลาร์สหรัฐ
3. ระบุวันสิ้นสุดปีงบประมาณของแต่ละปี
4. คำนวณอัตราการเติบโต พร้อมแสดงสูตรและการแทนค่า
5. แสดงอัตราการเติบโตทศนิยมสองตำแหน่ง
6. สรุปภาษาไทยสั้น ๆ ว่ารายได้เปลี่ยนแปลงอย่างไร
7. อ้างอิงแหล่งข้อมูลที่รองรับตัวเลข
8. หากเปิดแหล่งข้อมูลไม่ได้หรือข้อมูลไม่พอ ให้ระบุข้อจำกัด
9. ห้ามเดาตัวเลข

ตอบเป็นตาราง ตามด้วยสูตร การคำนวณ และข้อสรุป
ต้องค้นเว็บและเปิดอ่านแหล่งข้อมูลจริงก่อนตอบ
""".strip()


In [ ]:
web_001_run = begin_run(CONFIG, 'web_native_001', native_web_prompt, native_web_system_prompt,
    category='web', tools=[], fixtures={},
    case_version='2.0', fixture_version=None,
    timeout_s=180, source_recursion_limit=30,
    max_model_requests_per_turn=40)

print("New experiment:", web_001_run['run_id'])


In [ ]:
unsupported_web(web_001_run, CAPABILITIES['web_capability'])
show(web_001_run)


### Web 002

In [ ]:
web_002_prompt = """
ค้นเว็บเพื่อเปรียบเทียบ Microsoft กับ Alphabet
โดยใช้ผลประกอบการรวมทั้งปีงบประมาณ 2024 ของแต่ละบริษัท

ขั้นตอนที่ต้องทำ:
- ค้นหาแหล่งข้อมูลทางการของบริษัท หรือเอกสารที่ยื่นต่อ SEC
- เปิดอ่านแหล่งข้อมูลของแต่ละบริษัท อย่างน้อยบริษัทละหนึ่ง URL
  รวมอย่างน้อยสอง URL ที่แตกต่างกัน
- ตรวจตัวเลขจากเนื้อหาที่เปิดอ่าน ไม่ใช้เพียงข้อความย่อผลค้นหา

สิ่งที่ต้องหา:
1. วันเริ่มต้นและวันสิ้นสุดปีงบประมาณ
2. รายได้รวม
3. กำไรจากการดำเนินงาน (Operating income)
4. Operating margin ที่คำนวณจากตัวเลขข้างต้น

ข้อกำหนด:
- ใช้ข้อมูล GAAP จากงบรวมทั้งปี ไม่ใช้รายไตรมาสหรือราย segment
- ใช้แหล่งข้อมูลทางการของบริษัท หรือเอกสารที่บริษัทยื่นต่อ SEC
- แสดงจำนวนเงินเป็นล้านดอลลาร์สหรัฐ
- คำนวณ Operating margin เป็นเปอร์เซ็นต์ ทศนิยมสองตำแหน่ง
- แสดงสูตรและการแทนค่าของแต่ละบริษัท
- อ้างอิงให้ชัดว่าแต่ละแหล่งรองรับบริษัทใด
- ตรวจว่าช่วงปีงบประมาณตรงกันหรือไม่ และอธิบายผลต่อการเปรียบเทียบ
- สรุปว่าบริษัทใดมี Operating margin สูงกว่าในงบที่นำมาเทียบ
- ไม่สรุปว่าบริษัทนั้นดีกว่าทุกด้านจากตัวชี้วัดนี้เพียงตัวเดียว
- หากเปิดแหล่งข้อมูลไม่ได้หรือข้อมูลไม่พอ ให้ระบุข้อจำกัด
- ห้ามเดาตัวเลข

ตอบภาษาไทย ประกอบด้วย:
1. ตารางเปรียบเทียบ
2. สูตรและการคำนวณ
3. ข้อสรุปและข้อจำกัดสั้น ๆ

ต้องค้นเว็บและเปิดอ่านแหล่งข้อมูลจริงก่อนตอบ
""".strip()


In [ ]:
web_002_run = begin_run(CONFIG, 'web_native_002', web_002_prompt, native_web_system_prompt,
    category='web', tools=[], fixtures={},
    case_version='2.0', fixture_version=None,
    timeout_s=180, source_recursion_limit=30,
    max_model_requests_per_turn=40)

print("New experiment:", web_002_run['run_id'])


In [ ]:
unsupported_web(web_002_run, CAPABILITIES['web_capability'])
show(web_002_run)


### บันทึกท้ายหมวด

บันทึกได้ทุกสถานะ รวม runtime error และ pending review; export ซ้ำคง run_id เดิม แต่สร้างไฟล์ใหม่

In [ ]:
web_export = export_results('web', [web_001_run, web_002_run])
print(web_export)


# 5.Planning

`write_todos` เป็น custom tool: content + status pending/in_progress/completed และแทนรายการทั้งหมด
เครื่องมือไม่สร้างแผนให้ Pi ไม่ย้าย TodoListMiddleware หรือ built-in instructions ของ Deep Agents มาใช้
ไม่มีข้อห้าม multiple write_todos ในคำตอบเดียวแบบ middleware เดิม; บันทึกเป็นความต่างของ harness
ตรวจจากแผนที่เขียนและ tool trace เท่านั้น; ความสมเหตุสมผลของแผน/คำตอบต้อง review รอบใหม่

In [ ]:
PLANNING_DATA = {
    "co-d81": {
        "company_id": "co-d81",
        "company_name": "Delta",
        "year": 2025,
        "previous_year": 2024,
        "unit": "million_thb",
        "revenue_previous": 1000,
        "revenue_current": 1300,
        "net_profit_current": 156,
        "liabilities": 540,
        "assets": 1800,
        "source_id": "fixture_delta_v1",
    },
    "co-e42": {
        "company_id": "co-e42",
        "company_name": "Echo",
        "year": 2025,
        "previous_year": 2024,
        "unit": "million_thb",
        "revenue_previous": 800,
        "revenue_current": 960,
        "net_profit_current": 192,
        "liabilities": 900,
        "assets": 1500,
        "source_id": "fixture_echo_v1",
    },
    "co-f63": {
        "company_id": "co-f63",
        "company_name": "Foxtrot",
        "year": 2025,
        "previous_year": 2024,
        "unit": "million_thb",
        "revenue_previous": 1500,
        "revenue_current": 1800,
        "net_profit_current": None,
        "liabilities": 400,
        "assets": 2000,
        "source_id": "fixture_foxtrot_v1",
    },
}


In [ ]:
planning_tools = tool_specs(['write_todos', 'list_companies', 'read_company_financials', 'calculate_percentage'])
print(json.dumps(planning_tools, ensure_ascii=False, indent=2))


In [ ]:
planning_prompt = """
เปรียบเทียบบริษัททั้งหมดที่มีในระบบ โดยใช้ข้อมูลปี 2025
ในสามด้าน:
- การเติบโตของรายได้เทียบปีก่อน
- อัตรากำไรสุทธิ
- หนี้สินรวมต่อสินทรัพย์รวม

ระบุว่าบริษัทใดมีข้อมูลครบสำหรับคำนวณทั้งสามตัวชี้วัด
และบริษัทใดยังขาดข้อมูลอะไร

ข้อกำหนด:
- สร้างแผนงานของคุณผ่าน write_todos ก่อนเริ่มเรียก tools ดึงข้อมูล
- เลือกงานย่อยและลำดับการทำงานเอง
- อัปเดตสถานะ To-do ระหว่างทำงานให้ตรงกับสิ่งที่ทำจริง
- ใช้ calculate_percentage สำหรับการคำนวณ
- ข้อมูลขาดให้แสดง N/A และอธิบาย ห้ามสมมติตัวเลข
- ข้อมูลครบ หมายถึงมีข้อมูลสำหรับคำนวณทั้งสามตัวชี้วัด
- อ้างอิง source_id สำหรับข้อมูลของแต่ละบริษัท
- ตอบภาษาไทยเป็นตาราง พร้อมข้อสรุปสั้น ๆ
"""


In [ ]:
planning_instructions = (
"Complete the user's analysis using the available tools. "
        "Maintain an actionable task list reflecting actual progress. "
        "Use only the supplied financial data. "
        "Treat missing values as unavailable, not as zero. "
        "For this experiment, perform the work yourself without delegation."
)


In [ ]:
planning_001_run = begin_run(CONFIG, 'planning_001', planning_prompt, planning_instructions,
    category='planning', tools=planning_tools, fixtures={'PLANNING_DATA': PLANNING_DATA},
    case_version=None, fixture_version=None,
    timeout_s=180, source_recursion_limit=80,
    max_model_requests_per_turn=40)
planning_001_run['implementation_differences'].append('Custom write_todos replaces full list; no Deep Agents TodoListMiddleware instructions or multi-call guard')
print("New experiment:", planning_001_run['run_id'])


In [ ]:
run_once(planning_001_run)
print("Runtime:", planning_001_run["runtime_status"])


In [ ]:
score(planning_001_run, None, tolerance=0, review_criteria=('plan_is_actionable_and_progress_matches_evidence', 'metrics_and_calculation_arguments_correct', 'sources_and_missing_data_correct', 'requested_thai_table_and_completeness'))
show(planning_001_run)


### Planning 002 — ปรับแผนเมื่อ primary source ใช้ไม่ได้

In [ ]:
echo_backup_2025 = deepcopy(PLANNING_DATA["co-e42"])


In [ ]:
echo_backup_2025["source_id"] = "fixture_echo_backup_2025"


In [ ]:
echo_backup_2024 = {
    **deepcopy(PLANNING_DATA["co-e42"]),
    "year": 2024,
    "previous_year": 2023,
    "revenue_previous": 700,
    "revenue_current": 800,
    "net_profit_current": 128,
    "liabilities": 880,
    "assets": 1500,
    "source_id": "fixture_echo_backup_2024",
}

BACKUP_SOURCES = {
    "fixture_echo_backup_2024": echo_backup_2024,
    "fixture_echo_backup_2025": echo_backup_2025,
}


In [ ]:
planning_002_tools = tool_specs(['write_todos', 'list_companies', 'read_primary_financials', 'find_financial_sources', 'read_financial_source', 'calculate_percentage'])


In [ ]:
planning_002_prompt = """
เปรียบเทียบบริษัททั้งหมดในระบบ โดยใช้ข้อมูลปี 2025
ในสามด้าน:
- การเติบโตของรายได้เทียบปีก่อน
- อัตรากำไรสุทธิ
- หนี้สินรวมต่อสินทรัพย์รวม

ระบุว่าบริษัทใดมีข้อมูลครบสำหรับคำนวณทั้งสามตัวชี้วัด
และบริษัทใดยังขาดข้อมูลอะไร

ข้อกำหนด:
- สร้างแผนผ่าน write_todos ก่อนเริ่มดึงข้อมูล
- เลือกงานย่อยและลำดับการทำงานเอง
- เริ่มตรวจข้อมูลจากแหล่งหลัก
- เมื่อพบอุปสรรค ให้ประเมินทางเลือกและอัปเดตแผนก่อนดำเนินทางเลือกนั้น
- ไม่เรียกซ้ำโดยไม่มีเหตุผลเมื่อ error ระบุ retryable เป็น false
- ตรวจบริษัทและปีของเอกสารให้ตรงกับงานก่อนนำตัวเลขมาใช้
- ใช้ calculate_percentage สำหรับการคำนวณ
- ข้อมูลขาดให้แสดง N/A และอธิบาย ห้ามสมมติตัวเลข
- อ้างอิง source_id ของข้อมูลที่ใช้จริง
- ตอบภาษาไทยเป็นตาราง พร้อมข้อสรุปสั้น ๆ
- อธิบายว่าพบอุปสรรคอะไรและแก้ไขอย่างไร
"""


In [ ]:
planning_002_instructions = (
"Complete the analysis using the supplied financial tools. "
        "Maintain an actionable task list reflecting actual progress. "
        "Revise your plan when new evidence requires a different approach. "
        "Treat missing values as unavailable, not as zero. "
        "Perform the work yourself without delegation."
)


In [ ]:
planning_002_run = begin_run(CONFIG, 'planning_002', planning_002_prompt, planning_002_instructions,
    category='planning', tools=planning_002_tools, fixtures={'PLANNING_DATA': PLANNING_DATA, 'BACKUP_SOURCES': BACKUP_SOURCES},
    case_version=None, fixture_version='planning_backup_v1',
    timeout_s=180, source_recursion_limit=100,
    max_model_requests_per_turn=40)
planning_002_run['implementation_differences'].append('Custom write_todos; no original middleware prompt or multiple-plan-call guard')
print("New experiment:", planning_002_run['run_id'])


In [ ]:
run_once(planning_002_run)
print("Runtime:", planning_002_run["runtime_status"])


In [ ]:
score(planning_002_run, None, tolerance=0, review_criteria=('plan_is_actionable_and_progress_matches_evidence', 'metrics_and_calculation_arguments_correct', 'sources_and_missing_data_correct', 'requested_thai_table_and_completeness', 'revised_plan_addresses_primary_failure', 'selected_backup_matches_company_and_reporting_year'))
show(planning_002_run)


### Review จากหลักฐานรอบใหม่

ดูคำตอบและ trace แล้วกรอกเฉพาะเกณฑ์ที่ตรวจแล้วด้วย `apply_review`
ไม่มีค่า True/False ที่กรอกไว้ล่วงหน้า; ถ้ายังไม่ตรวจให้ export แบบ pending_review ได้
ฟังก์ชันผูก review กับ run_id และ evidence hash อัตโนมัติ; เปลี่ยนคำตอบ/trace แล้ว review เก่าใช้ไม่ได้

In [ ]:
# ตัวอย่างวิธีเรียก หลังตรวจจริงแล้วเท่านั้น:
# apply_review(planning_001_run, checks={...}, notes='ระบุ call IDs/ข้อความที่ตรวจ', reviewer='ชื่อผู้ตรวจ')
# show(planning_001_run)
print('Planning 001 evidence:', evidence_hash(planning_001_run))
print('Planning 002 evidence:', evidence_hash(planning_002_run))


### บันทึกท้ายหมวด

บันทึกได้ทุกสถานะ รวม runtime error และ pending review; export ซ้ำคง run_id เดิม แต่สร้างไฟล์ใหม่

In [ ]:
planning_export = export_results('planning', [planning_001_run, planning_002_run])
print(planning_export)


# 6.Long context

คัดข้อความและลำดับเอกสารตรงต้นฉบับ: 60 และ 300 distractors
ตรวจ SHA-256 ของ UTF-8 ไม่ใช้แค่จำนวนตัวอักษร; Python len นับ code points ส่วน JS length นับ UTF-16 units
ใช้ prompt เดียวกันทั้งสอง runtime โดย JSON ไม่ตัดหรือสรุปเอกสาร และไม่มี tools

In [ ]:
def make_background(start, stop):
    return "\n\n".join(
        (
            f"[บันทึกประกอบ B{i:03d}]\n"
            f"บริษัท Other-{i:03d} รายงานยอดขายปี 2025 "
            f"จำนวน {2000 + i * 17} ล้านบาท "
            f"และกำไรสุทธิ {100 + i} ล้านบาท "
            "ฝ่ายปฏิบัติการอยู่ระหว่างทบทวนแผนคลังสินค้า "
            "ข้อมูลนี้เป็นของบริษัทอื่น ไม่ใช่งบของบริษัทเป้าหมาย"
        )
        for i in range(start, stop)
    )

long_documents = "\n\n".join([
    """
[เอกสาร N1]
บริษัท Northstar | ปี 2025 | งบรวม | หน่วย ล้านบาท
รายได้ 1200 | กำไรสุทธิ 120

[เอกสาร O1]
บริษัท Orchid | ปี 2025 | งบรวม | หน่วย ล้านบาท
รายได้ 900 | กำไรสุทธิ 90

[เอกสาร P1]
บริษัท Pine | ปี 2025 | งบรวม | หน่วย ล้านบาท
รายได้ 1500 | กำไรสุทธิ 150
""",
    make_background(1, 21),
    """
[ประกาศแก้ไข N2 | revision 2]
แทนที่เอกสาร N1 ทั้งฉบับ
บริษัท Northstar | ปี 2025 | งบรวม | หน่วย ล้านบาท
รายได้ 1250 | กำไรสุทธิ 125

[เอกสาร O-SEPARATE]
บริษัท Orchid | ปี 2025 | งบเฉพาะกิจการ | หน่วย ล้านบาท
รายได้ 700 | กำไรสุทธิ 140
""",
    make_background(21, 41),
    """
[ประกาศแก้ไข P2 | revision 2]
แทนที่เอกสาร P1 ทั้งฉบับ
บริษัท Pine | ปี 2025 | งบรวม | หน่วย ล้านบาท
รายได้ 1500 | กำไรสุทธิ: ไม่เปิดเผย
ห้ามนำกำไรสุทธิจากฉบับที่ถูกแทนที่มาเติม

[ประกาศแก้ไข N3 | revision 3]
แทนที่เอกสาร N2 ทั้งฉบับ
บริษัท Northstar | ปี 2025 | งบรวม | หน่วย ล้านบาท
รายได้ 1300 | กำไรสุทธิ 156
""",
    make_background(41, 61),
    """
[เอกสาร O-2024]
บริษัท Orchid | ปี 2024 | งบรวม | หน่วย ล้านบาท
รายได้ 800 | กำไรสุทธิ 160

[สำเนาประกาศเก่าสำหรับการตรวจสอบย้อนหลัง]
เอกสาร N2 | revision 2
บริษัท Northstar | ปี 2025 | งบรวม | หน่วย ล้านบาท
รายได้ 1250 | กำไรสุทธิ 125
สำเนานี้ไม่ใช่ revision ใหม่
""",
])


In [ ]:
long_system_prompt = (
    "Read the supplied documents and follow the user's output schema. "
    "Use only information in the user message. "
    "Do not call tools, browse, delegate, or access files. "
    "Return only the requested JSON object."
)

long_prompt = """
อ่านเอกสารทั้งหมดแล้วสรุปเฉพาะ Northstar, Orchid และ Pine

ข้อกำหนด:
1. ใช้เฉพาะงบรวมปี 2025 หน่วยล้านบาท
2. เรียงบริษัทตามนี้เท่านั้น: Northstar, Orchid, Pine
3. หากมีประกาศแทนที่ ให้เลือก revision สูงสุดของงบที่ตรงเงื่อนไข
   ห้ามเลือกเพียงเพราะข้อความนั้นอยู่ท้ายเอกสาร
4. ประกาศแทนที่มีผลทั้งฉบับ ห้ามผสมข้อมูลกับฉบับเก่า
5. หากกำไรสุทธิไม่เปิดเผย ให้ใช้ null และ missing_fields=["net_profit_million"]
   หากเปิดเผย ให้ missing_fields=[]
6. source_id ต้องเป็นรหัสเอกสารที่ใช้จริง
7. ห้ามใช้ tools ทุกชนิด ใช้ข้อมูลในคำถามนี้เท่านั้น
8. ตอบ JSON object เท่านั้น ไม่ใส่ code fence หรือคำอธิบาย
9. ห้ามเพิ่ม keys นอกโครงสร้างที่กำหนด
10. ตัวเลขต้องเป็น JSON number ไม่ใช่ string

โครงสร้าง:
{
  "year": 2025,
  "unit": "million_thb",
  "companies": [
    {
      "company_name": "ชื่อบริษัท",
      "revenue_million": 0,
      "net_profit_million": null,
      "source_id": "รหัสเอกสาร",
      "missing_fields": []
    }
  ]
}

รายการ companies ต้องมีครบ 3 บริษัทตามลำดับที่กำหนด

เอกสาร:
""" + long_documents


In [ ]:
long_001_run = begin_run(CONFIG, 'long_context_001', long_prompt, long_system_prompt,
    category='long_context', tools=[], fixtures={},
    case_version=None, fixture_version='long_context_v1',
    timeout_s=180, source_recursion_limit=20,
    max_model_requests_per_turn=40)

print("New experiment:", long_001_run['run_id'])


In [ ]:
print('UTF-8 SHA-256:', digest(long_prompt), 'Python chars:', len(long_prompt))


In [ ]:
run_once(long_001_run)
print("Runtime:", long_001_run["runtime_status"])


In [ ]:
expected_long_rows = [
    {
        "company_name": "Northstar",
        "revenue_million": 1300,
        "net_profit_million": 156,
        "source_id": "N3",
        "missing_fields": [],
    },
    {
        "company_name": "Orchid",
        "revenue_million": 900,
        "net_profit_million": 90,
        "source_id": "O1",
        "missing_fields": [],
    },
    {
        "company_name": "Pine",
        "revenue_million": 1500,
        "net_profit_million": None,
        "source_id": "P2",
        "missing_fields": ["net_profit_million"],
    },
]


In [ ]:
expected_long = {'year': 2025, 'unit': 'million_thb', 'companies': expected_long_rows}


In [ ]:
score(long_001_run, expected_long, tolerance=0, review_criteria=())
show(long_001_run)


### Long context 002

In [ ]:
remaining = long_documents
document_parts = []

for start, stop in [(1, 21), (21, 41), (41, 61)]:
    old_background = make_background(start, stop)
    before, separator, remaining = remaining.partition(old_background)

    assert separator, f"ไม่พบข้อมูลรบกวนช่วง {start}–{stop - 1}"
    document_parts.append(before)

document_parts.append(remaining)

# เพิ่มจาก 60 เป็น 300 บันทึก โดยใช้หมายเลขไม่ซ้ำ
long_002_documents = (
    document_parts[0]
    + make_background(1, 101)
    + document_parts[1]
    + make_background(101, 201)
    + document_parts[2]
    + make_background(201, 301)
    + document_parts[3]
)

# ใช้คำสั่งเดิม เปลี่ยนเฉพาะเอกสาร
assert long_prompt.endswith(long_documents)

instruction_prefix = long_prompt[:-len(long_documents)]
long_002_prompt = instruction_prefix + long_002_documents

print("Original prompt characters:", f"{len(long_prompt):,}")
print("New prompt characters:", f"{len(long_002_prompt):,}")
print("Length ratio:", round(len(long_002_prompt) / len(long_prompt), 2))
print("Distractor records: 60 → 300")


In [ ]:
long_002_run = begin_run(CONFIG, 'long_context_002', long_002_prompt, long_system_prompt,
    category='long_context', tools=[], fixtures={},
    case_version=None, fixture_version='long_context_v1_distractors_300',
    timeout_s=180, source_recursion_limit=20,
    max_model_requests_per_turn=40)

print("New experiment:", long_002_run['run_id'])


In [ ]:
print('UTF-8 SHA-256:', digest(long_002_prompt), 'Python chars:', len(long_002_prompt))


In [ ]:
run_once(long_002_run)
print("Runtime:", long_002_run["runtime_status"])


In [ ]:
score(long_002_run, expected_long, tolerance=0, review_criteria=())
show(long_002_run)


### บันทึกท้ายหมวด

บันทึกได้ทุกสถานะ รวม runtime error และ pending review; export ซ้ำคง run_id เดิม แต่สร้างไฟล์ใหม่

In [ ]:
long_context_export = export_results('long_context', [long_001_run, long_002_run])
print(long_context_export)


# 7.Session history

ใช้ native `SessionManager` JSONL ของ Pi สำหรับทั้งสามเคส
history_001 ทำ seed และ recall ใน Node process เดียว; history_002 เปิดไฟล์เดิมใน process ใหม่แล้วแก้ไข/recall
ต่างจาก in-memory checkpointer ของ Deep Agents และบันทึก backend ตามจริง
usage นับเฉพาะ assistant responses ใหม่ ไม่รวมข้อความที่โหลดกลับมา

In [ ]:
history_system_prompt = (
    "You are a session-memory test agent. "
    "Use only information from the current conversation thread. "
    "When asked for JSON, return JSON only without Markdown."
)


In [ ]:
history_turn_1_prompt = """
จดจำข้อมูลต่อไปนี้ไว้สำหรับคำถามถัดไปใน session นี้:

บริษัท: Aurora
ปีงบประมาณ: 2025
รายได้รวม: 2400 ล้านบาท
กำไรสุทธิ: 288 ล้านบาท
อัตรากำไรสุทธิ: 12%

ตอบเพียงคำว่า CONTEXT_SAVED
"""


In [ ]:
history_turn_2_prompt = """
จากข้อมูลที่บันทึกไว้ก่อนหน้านี้ ตอบเป็น JSON object เท่านั้น:

{
  "company": "ชื่อบริษัท",
  "year": 0,
  "revenue_million": 0,
  "net_profit_million": 0,
  "net_margin_pct": 0
}

ห้ามเพิ่ม key อื่น และห้ามคำนวณหรือเดาตัวเลขใหม่
"""


In [ ]:
history_001_run = begin_run(CONFIG, 'history_001', history_turn_2_prompt, history_system_prompt,
    category='history', tools=[], fixtures={},
    case_version=None, fixture_version='session_history_v1',
    timeout_s=120, source_recursion_limit=None,
    max_model_requests_per_turn=40)
history_001_run['prompts'] = [history_turn_1_prompt, history_turn_2_prompt]
history_001_run['persistence'] = True
print("New experiment:", history_001_run['run_id'])


In [ ]:
run_once(history_001_run)
print("Runtime:", history_001_run["runtime_status"])


In [ ]:
expected_history = {
    "company": "Aurora",
    "year": 2025,
    "revenue_million": 2400,
    "net_profit_million": 288,
    "net_margin_pct": 12,
}


In [ ]:
score(history_001_run, expected_history, tolerance=0, review_criteria=())
show(history_001_run)


### History 002 — ข้อมูลแก้ไขล่าสุด

In [ ]:
history_correction_prompt = """
แก้ไขข้อมูลของบริษัทที่เราคุยกันก่อนหน้านี้:

- กำไรสุทธิที่ถูกต้องคือ 360 ล้านบาท แทนค่าเดิม
- อัตรากำไรสุทธิที่ถูกต้องคือ 15% แทนค่าเดิม
- ชื่อบริษัท ปีงบประมาณ และรายได้รวม คงเดิม

จากนี้ให้ใช้ค่าที่แก้ไขแล้ว
ตอบเพียงคำว่า CONTEXT_UPDATED
"""

history_recall_prompt = """
สรุปข้อมูลล่าสุดของบริษัทที่เราคุยกันไว้
ใช้ค่าที่แก้ไขล่าสุด และคงข้อมูลที่ไม่ได้ถูกแก้ไข

ตอบ JSON object เท่านั้น มี keys ดังนี้:
company, year, revenue_million, net_profit_million, net_margin_pct

ห้ามเพิ่ม key อื่น ห้ามใช้ tools และห้ามเดาข้อมูล
"""


In [ ]:
history_002_run = begin_run(CONFIG, 'history_002', history_recall_prompt, history_system_prompt,
    category='history', tools=[], fixtures={},
    case_version=None, fixture_version='session_history_correction_v1',
    timeout_s=120, source_recursion_limit=20,
    max_model_requests_per_turn=40)
history_002_run['prompts'] = [history_correction_prompt, history_recall_prompt]
history_002_run['persistence'] = True
history_002_run['parent_run_id'] = history_001_run['run_id']
for key in ('session_file', 'session_sha256'):
    history_002_run[key] = history_001_run.get(key)
history_002_run['previous_node_pid'] = history_001_run.get('node_pid')
history_002_run['expected_versions'] = history_001_run.get('versions')
print("New experiment:", history_002_run['run_id'])


In [ ]:
if not history_002_run.get('session_file'):
    history_002_run.update(runtime_status='runtime_error', error={'type': 'MissingSession', 'message': 'history_001 did not persist a session'})
else:
    run_once(history_002_run)


In [ ]:
expected_h2 = {
    "company": "Aurora",
    "year": 2025,
    "revenue_million": 2400,
    "net_profit_million": 360,
    "net_margin_pct": 15,
}


In [ ]:
score(history_002_run, expected_h2, tolerance=0, review_criteria=())
show(history_002_run)


### บันทึกท้ายหมวด

บันทึกได้ทุกสถานะ รวม runtime error และ pending review; export ซ้ำคง run_id เดิม แต่สร้างไฟล์ใหม่

In [ ]:
history_export = export_results('history', [history_001_run, history_002_run])
print(history_export)


### History 003 — Seed ก่อน Restart Kernel

In [ ]:
restart_system_prompt = (
    "Use only information from the current conversation thread. "
    "Do not call tools or access files. "
    "Follow the requested response format exactly."
)

restart_seed_prompt = """
จดจำข้อมูลสำหรับการสนทนานี้:

บริษัท: Vega
ปีงบประมาณ: 2025
รายได้รวม: 3200 ล้านบาท
กำไรสุทธิ: 480 ล้านบาท
อัตรากำไรสุทธิ: 15%
รหัสอ้างอิง: VEGA-73-KP

ตอบเพียงคำว่า CONTEXT_SAVED
"""


In [ ]:
history_003_seed = begin_run(CONFIG, 'history_003', restart_seed_prompt, restart_system_prompt,
    category='history', tools=[], fixtures={},
    case_version=None, fixture_version=None,
    timeout_s=120, source_recursion_limit=20,
    max_model_requests_per_turn=40)
history_003_seed['persistence'] = True
print("New experiment:", history_003_seed['run_id'])


In [ ]:
run_once(history_003_seed)
print("Runtime:", history_003_seed["runtime_status"])


In [ ]:
# ไม่ assert ว่าคำตอบต้องผ่านก่อนบันทึก error
if history_003_seed['runtime_status'] == 'completed':
    restart_manifest_path = save_restart_manifest(history_003_seed, KERNEL_IDENTITY)
    print('บันทึก path นี้ไว้เพื่อเลือกหลัง restart:', restart_manifest_path)
else:
    show(history_003_seed)
print(export_results('history', [history_003_seed]))


## หยุดตรงนี้ → Kernel → Restart Kernel

หลัง restart ให้รันเฉพาะสองเซลล์ใน **0.Config** แล้วข้ามไปเซลล์ด้านล่าง
ห้ามรัน Smoke, history seed หรือเซลล์หมวดอื่นซ้ำก่อน resume
Resume ตรวจ Python PID, Node PID, hash ของ native session และ versions ก่อน model call
prompt หลัง restart มีเฉพาะคำถามเดิม ไม่ส่ง seed/expected answer ซ้ำ

In [ ]:
raise RuntimeError('Restart Kernel แล้วรัน Config ก่อนข้ามไปเซลล์เลือก manifest ด้านล่าง')


### กู้ session หลัง Restart

In [ ]:
restart_candidates = sorted((shared_root() / 'state' / SDK).glob('*/restart-manifest_*.json'),
                            key=lambda p: p.stat().st_mtime)
for index, candidate in enumerate(restart_candidates):
    manifest_info = json.loads(candidate.read_text(encoding='utf-8'))
    print(index, manifest_info['run_id'], candidate)
if not restart_candidates:
    raise FileNotFoundError('ไม่พบ Pi restart manifest; ต้อง seed ก่อน')
MANIFEST_INDEX = -1  # แก้ index หากต้องการ session รอบอื่น; default คือ seed ล่าสุด
selected_manifest = restart_candidates[MANIFEST_INDEX]
print('Selected:', selected_manifest)


In [ ]:
resume_prompt = """
จากข้อมูลที่ให้ไว้ก่อนหน้านี้ในบทสนทนานี้
ตอบ JSON object เท่านั้น โดยมี keys:
company, year, revenue_million, net_profit_million,
net_margin_pct, reference_code

ใช้ข้อมูลและรหัสอ้างอิงจากประวัติเดิม
ห้ามเดา ห้ามเพิ่ม keys และห้ามใช้ tools
หากไม่พบข้อมูลของ field ใด ให้ใช้ null
"""


In [ ]:
history_003_run = resume_from_manifest(selected_manifest, CONFIG, KERNEL_IDENTITY, resume_prompt)


In [ ]:
expected_resume = {
    "company": "Vega",
    "year": 2025,
    "revenue_million": 3200,
    "net_profit_million": 480,
    "net_margin_pct": 15,
    "reference_code": "VEGA-73-KP",
}


In [ ]:
score(history_003_run, expected_resume, tolerance=0, review_criteria=())
show(history_003_run)


### บันทึกท้ายหมวด

บันทึกได้ทุกสถานะ รวม runtime error และ pending review; export ซ้ำคง run_id เดิม แต่สร้างไฟล์ใหม่

In [ ]:
history_export = export_results('history', [history_003_run])
print(history_export)


# 8.Error handling

Business errors เป็นผลเครื่องมือปกติที่มี retryable flag; ไม่ใช่ API retries
nonretryable ต้องไม่เรียกซ้ำ; retryable เคสนี้อนุญาต 2 ครั้งรวมครั้งแรก และครั้งที่สองต้องหลังได้รับ error แรก

### Error 001

In [ ]:
error_system_prompt = """
Use the supplied tool to retrieve financial statements.
If the tool returns an error, follow its retryable flag.
Do not invent financial data.
Do not delegate or use other tools.
Return only the JSON requested by the user.
"""

error_prompt = """
ขอดูงบปี 2025 ของบริษัท Aurora โดยเรียกเครื่องมือที่ให้

ข้อกำหนด:
- หากเครื่องมือคืน retryable=false ห้ามเรียกคำขอนั้นซ้ำ
- หากดึงข้อมูลไม่ได้ ให้รายงานข้อผิดพลาดตามจริง
- ห้ามใช้ตัวเลขจากความรู้เดิมหรือเดาตัวเลข
- ตอบ JSON object เท่านั้น มี keys ตามนี้:

{
  "status": "ok หรือ unavailable",
  "company_name": "Aurora",
  "year": 2025,
  "error_code": "รหัสข้อผิดพลาดจากเครื่องมือ หรือ null เมื่อสำเร็จ",
  "statement": "ข้อมูลงบจากเครื่องมือ หรือ null เมื่อไม่สำเร็จ"
}
"""


In [ ]:
error_001_tools = tool_specs(['fetch_restricted_statement'])


In [ ]:
error_001_run = begin_run(CONFIG, 'error_001', error_prompt, error_system_prompt,
    category='error', tools=error_001_tools, fixtures={},
    case_version=None, fixture_version='nonretryable_error_v1',
    timeout_s=120, source_recursion_limit=12,
    max_model_requests_per_turn=40)

print("New experiment:", error_001_run['run_id'])


In [ ]:
run_once(error_001_run)
print("Runtime:", error_001_run["runtime_status"])


In [ ]:
expected_error_answer = {
    "status": "unavailable",
    "company_name": "Aurora",
    "year": 2025,
    "error_code": "ACCESS_DENIED",
    "statement": None,
}


In [ ]:
score(error_001_run, expected_error_answer, tolerance=0, review_criteria=())
show(error_001_run)


### Error 002

In [ ]:
retry_limit_system_prompt = """
Use the supplied tool to retrieve financial statements.
Follow the retry limit specified by the user.
Do not invent financial data.
Do not delegate or use other tools.
Return only the requested JSON.
"""

retry_limit_prompt = """
ขอดูงบปี 2025 ของบริษัท Aurora โดยเรียกเครื่องมือที่ให้

นโยบายจัดการข้อผิดพลาด:
- หากเครื่องมือคืน retryable=true ให้ลองคำขอเดิมซ้ำอีก 1 ครั้ง
- อนุญาตให้เรียกเครื่องมือรวมสูงสุด 2 ครั้ง
- หากครั้งที่สองยังผิดพลาด ต้องหยุดและรายงานว่าไม่สามารถดึงข้อมูลได้
- ห้ามเดาตัวเลขหรือใช้ข้อมูลจากความรู้เดิม

ตอบ JSON object เท่านั้น มี keys:
{
  "status": "ok หรือ unavailable",
  "company_name": "Aurora",
  "year": 2025,
  "error_code": "รหัสข้อผิดพลาดล่าสุด หรือ null เมื่อสำเร็จ",
  "attempts": 0,
  "statement": "ข้อมูลงบ หรือ null เมื่อไม่สำเร็จ"
}

attempts ต้องเป็นจำนวนครั้งที่เรียกเครื่องมือจริง
"""


In [ ]:
error_002_tools = tool_specs(['fetch_temporarily_unavailable_statement'])


In [ ]:
error_002_run = begin_run(CONFIG, 'error_002', retry_limit_prompt, retry_limit_system_prompt,
    category='error', tools=error_002_tools, fixtures={},
    case_version=None, fixture_version='retry_exhaustion_v1',
    timeout_s=120, source_recursion_limit=16,
    max_model_requests_per_turn=40)

print("New experiment:", error_002_run['run_id'])


In [ ]:
run_once(error_002_run)
print("Runtime:", error_002_run["runtime_status"])


In [ ]:
expected_retry_answer = {
    "status": "unavailable",
    "company_name": "Aurora",
    "year": 2025,
    "error_code": "TEMPORARY_UNAVAILABLE",
    "attempts": 2,
    "statement": None,
}


In [ ]:
score(error_002_run, expected_retry_answer, tolerance=0, review_criteria=())
show(error_002_run)


### บันทึกท้ายหมวด

บันทึกได้ทุกสถานะ รวม runtime error และ pending review; export ซ้ำคง run_id เดิม แต่สร้างไฟล์ใหม่

In [ ]:
error_export = export_results('error', [error_001_run, error_002_run])
print(error_export)


# 9.สรุปและสำรวจผล Benchmark

อ่านเฉพาะ `shared/results/pi_agent` แยก case_version และ deduplicate ตาม run_id
review revision ใหม่แทน revision เดิม; duplicate export ไม่เพิ่มจำนวนทดลอง
แสดง unsupported/runtime_error/pending_review แยกจาก failed; ไม่เปลี่ยน null เป็นศูนย์
summary บันทึกไฟล์ใหม่ใต้ shared/summaries โดยไม่เขียนทับผลเก่า

In [ ]:
summary_rows, summary_path = summarize()
print(json.dumps(summary_rows, ensure_ascii=False, indent=2))
print('Saved:', summary_path)
